# Word2Vec from Scratch — Skip-gram + Negative Sampling
**No `gensim`, no `torch`, no `autograd` — pure NumPy**

### What you'll build
A Word2Vec model that learns dense word embeddings from raw text by solving a self-supervised task:  
*given a center word, predict its surrounding context words.*

### Notebook structure
| # | Section | Key idea |
|---|---------|----------|
| 1 | Corpus & Tokenization | raw text → token list |
| 2 | Vocabulary | token counts, word↔index maps |
| 3 | Subsampling | discard frequent words stochastically |
| 4 | Skip-gram pairs | sliding window → (center, context) pairs |
| 5 | Negative Sampling Table | unigram^0.75 distribution |
| 6 | Model (forward + backward) | two embedding matrices, manual gradients |
| 7 | Training loop | SGD + linear LR decay |
| 8 | Evaluation | nearest neighbors, cosine sim, analogies |
| 9 | Visualisation | PCA 2-D plot of embeddings |

## 0 · Imports

In [ ]:
import numpy as np
import re
from collections import Counter
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

np.random.seed(42)
print("NumPy version:", np.__version__)

## 1 · Corpus & Tokenization

We use a hand-crafted corpus with clear semantic clusters:
- **royalty**: king, queen, prince, princess
- **animals**: dog, cat, puppy, kitten
- **capitals**: paris, berlin, london, rome → france, germany, england, italy
- **nature**: mountain, river, ocean, forest
- **tech**: machine learning, neural networks, algorithms

> **Why a custom corpus?**  
> With only ~300 tokens we can finish training in seconds and still observe meaningful geometry — useful for understanding mechanics before scaling up.

In [ ]:
RAW_CORPUS = """
the king rules the kingdom with wisdom and power
the queen rules beside the king with grace and wisdom
the prince will become the king one day
the princess will become the queen one day
man is the king of his domain
woman is the queen of her domain
the king and queen have a son the prince
the king and queen have a daughter the princess
paris is the capital of france
berlin is the capital of germany
london is the capital of england
rome is the capital of italy
france germany england italy are countries in europe
paris berlin london rome are cities in europe
the dog barks at the cat
the cat meows at the dog
the puppy is a young dog
the kitten is a young cat
dog and cat are common pets
the dog is loyal and friendly
the cat is independent and graceful
the river flows through the city
the mountain stands above the valley
the ocean is vast and deep
the forest is dense and green
the sun rises in the east
the moon shines at night
the star twinkles in the sky
scientists study the universe
doctors heal the sick
teachers educate the young
lawyers argue in court
engineers build bridges and roads
the car drives on the road
the train runs on the track
the plane flies in the sky
the ship sails on the ocean
fast cars race on the track
slow trains travel through the countryside
machine learning is a branch of artificial intelligence
deep learning uses neural networks
neural networks learn from data
algorithms process information efficiently
computers execute programs rapidly
"""

def tokenize(text: str) -> list:
    """Lowercase + keep only alphabetic tokens."""
    return re.findall(r'[a-z]+', text.lower())

tokens = tokenize(RAW_CORPUS)
print(f"Total tokens : {len(tokens)}")
print(f"Sample       : {tokens[:15]}")

## 2 · Vocabulary

Maps every unique word to an integer index and stores its frequency.  
Words appearing fewer than `min_count` times are dropped (they add noise, not signal).

In [ ]:
class Vocabulary:
    """
    Attributes
    ----------
    word2idx  : dict  word → integer index
    idx2word  : dict  integer index → word
    word_freq : dict  word → relative frequency (sums to 1)
    size      : int   vocabulary size
    """
    def __init__(self, tokens: list, min_count: int = 1):
        counts = Counter(tokens)
        counts = {w: c for w, c in counts.items() if c >= min_count}

        self.word2idx   = {}
        self.idx2word   = {}
        self.word_counts = {}

        for idx, (word, count) in enumerate(
            sorted(counts.items(), key=lambda x: -x[1])
        ):
            self.word2idx[word]  = idx
            self.idx2word[idx]   = word
            self.word_counts[word] = count

        self.size = len(self.word2idx)
        total = sum(self.word_counts.values())
        self.word_freq = {w: c / total for w, c in self.word_counts.items()}

    def __len__(self):
        return self.size


vocab = Vocabulary(tokens, min_count=1)
print(f"Vocabulary size : {vocab.size}")
print()
print("Top-15 most frequent words:")
for i in range(15):
    w = vocab.idx2word[i]
    print(f"  [{i:>3}]  {w:<15}  count={vocab.word_counts[w]}  freq={vocab.word_freq[w]:.4f}")

## 3 · Subsampling Frequent Words

**Problem**: "the", "is", "a" appear everywhere — they co-occur with every word, 
so training on every occurrence wastes compute without adding semantic signal.

**Word2Vec's fix** — discard each token stochastically with probability:

$$P(\text{discard}\ w) = 1 - \sqrt{\frac{t}{\text{freq}(w)}}$$

where $t \approx 10^{-3}$. High-frequency words get discarded ~80% of the time; rare words almost never.

> This is re-run **each epoch** so the kept pairs differ every time — a form of data augmentation.

In [ ]:
def subsample_tokens(tokens: list, vocab: Vocabulary, t: float = 1e-3) -> list:
    """
    Parameters
    ----------
    tokens : raw token list
    vocab  : Vocabulary (needs .word_freq)
    t      : subsampling threshold (default 1e-3, same as gensim)

    Returns
    -------
    Filtered token list with frequent words randomly removed.
    """
    result = []
    for w in tokens:
        if w not in vocab.word2idx:
            continue
        f = vocab.word_freq[w]
        # keep_prob → 1 for rare words, ~0 for very common words
        keep_prob = min(1.0, np.sqrt(t / f) + t / f)
        if np.random.random() < keep_prob:
            result.append(w)
    return result


# Show effect on common vs rare words
print("Subsampling keep-probabilities:")
print(f"  'the'        freq={vocab.word_freq.get('the',0):.4f}  keep≈{min(1., np.sqrt(1e-3/vocab.word_freq.get('the',1))):.2f}")
print(f"  'king'       freq={vocab.word_freq.get('king',0):.4f}  keep≈{min(1., np.sqrt(1e-3/vocab.word_freq.get('king',1))):.2f}")
print(f"  'princess'   freq={vocab.word_freq.get('princess',0):.4f}  keep≈{min(1., np.sqrt(1e-3/vocab.word_freq.get('princess',1))):.2f}")
print()

subsampled = subsample_tokens(tokens, vocab)
print(f"Original token count  : {len(tokens)}")
print(f"After subsampling     : {len(subsampled)}")

## 4 · Skip-gram Training Pairs

**Sliding window** — for each center word at position $i$, every word within 
$\pm$`window` positions forms a positive training pair.

```
tokens:   [the,  king,  rules,  the,  kingdom]
center:   'rules' (i=2), window=2
contexts: 'the'(i=1), 'king'(i=1), 'the'(i=3), 'kingdom'(i=4)
pairs:    (rules→the), (rules→king), (rules→the), (rules→kingdom)
```

**Dynamic windowing**: actual window is sampled uniformly from $[1, \text{window}]$ 
each time — gives closer words higher implicit weight in aggregate.

In [ ]:
def generate_skipgram_pairs(tokens: list, vocab: Vocabulary, window: int = 2) -> list:
    """
    Returns list of (center_idx, context_idx) integer pairs.
    Dynamic window: sample actual_window ~ Uniform[1, window] per center word.
    """
    pairs   = []
    indices = [vocab.word2idx[w] for w in tokens if w in vocab.word2idx]

    for i, center in enumerate(indices):
        actual_window = np.random.randint(1, window + 1)  # dynamic window
        start = max(0, i - actual_window)
        end   = min(len(indices), i + actual_window + 1)
        for j in range(start, end):
            if j != i:
                pairs.append((center, indices[j]))
    return pairs


pairs = generate_skipgram_pairs(subsampled, vocab, window=2)
print(f"Training pairs generated : {len(pairs)}")
print()
print("Sample pairs (word form):")
for center_idx, ctx_idx in pairs[:8]:
    print(f"  center='{vocab.idx2word[center_idx]}'  →  context='{vocab.idx2word[ctx_idx]}'")

## 5 · Negative Sampling Table

**Why not uniform sampling?**  
If we sampled negatives uniformly, very rare words would almost never appear — 
but common words would crowd out the useful signal.

**Word2Vec's solution** — sample proportional to $\text{freq}(w)^{0.75}$:

| Power | Effect |
|-------|--------|
| 1.0 (unigram) | Too peaked — common words dominate |
| 0.0 (uniform) | Too flat — no frequency weighting |
| **0.75** | Smooth compromise ✓ |

**Implementation**: fill a table of 1M slots; word $w$ gets $\propto \text{freq}(w)^{0.75}$ slots. 
Random index lookup = $O(1)$ sampling.

In [ ]:
class NegativeSampler:
    """
    Builds a large lookup table for O(1) negative sampling.
    Uses the smoothed unigram distribution: P(w) ∝ freq(w)^0.75
    """
    TABLE_SIZE = 1_000_000

    def __init__(self, vocab: Vocabulary):
        table = []
        total = sum(c ** 0.75 for c in vocab.word_counts.values())

        for word, count in vocab.word_counts.items():
            idx     = vocab.word2idx[word]
            n_slots = int((count ** 0.75 / total) * self.TABLE_SIZE)
            table.extend([idx] * n_slots)

        # Pad to exact TABLE_SIZE
        while len(table) < self.TABLE_SIZE:
            table.append(table[-1])

        self.table = np.array(table[:self.TABLE_SIZE], dtype=np.int32)

    def sample(self, n: int, exclude: int) -> np.ndarray:
        """Draw n negatives, never returning `exclude` index."""
        samples = []
        while len(samples) < n:
            idx = self.table[np.random.randint(0, self.TABLE_SIZE)]
            if idx != exclude:
                samples.append(idx)
        return np.array(samples)


neg_sampler = NegativeSampler(vocab)

# Visualise the distribution (compare uniform vs smoothed unigram)
freqs   = np.array([vocab.word_freq[vocab.idx2word[i]] for i in range(vocab.size)])
smooth  = freqs ** 0.75
smooth /= smooth.sum()

fig, axes = plt.subplots(1, 2, figsize=(12, 3))
axes[0].bar(range(vocab.size), freqs,  color='steelblue', width=1)
axes[0].set_title("Raw unigram distribution")
axes[0].set_xlabel("Word index (sorted by freq)")
axes[1].bar(range(vocab.size), smooth, color='coral',     width=1)
axes[1].set_title("Smoothed (^0.75) — used for neg sampling")
axes[1].set_xlabel("Word index")
plt.tight_layout()
plt.show()

## 6 · The Word2Vec Model

### Two embedding matrices

| Matrix | Shape | Role |
|--------|-------|------|
| `W_in`  | V × D | **Center** word embeddings — used after training |
| `W_out` | V × D | **Context** word embeddings — discarded after training |

Two matrices prevent the degenerate solution where a word is trivially its own best context.

### Loss function (negative sampling objective)

For positive pair $(w, c)$ and $k$ negatives $\{c_1^-, \ldots, c_k^-\}$:

$$\mathcal{L} = -\log \sigma(\mathbf{v}_w \cdot \mathbf{v}_c) - \sum_{i=1}^k \log \sigma(-\mathbf{v}_w \cdot \mathbf{v}_{c_i^-})$$

### Manual gradients

$$\frac{\partial \mathcal{L}}{\partial \mathbf{v}_w} = (\sigma(\mathbf{v}_w\cdot\mathbf{v}_c)-1)\,\mathbf{v}_c + \sum_i \sigma(\mathbf{v}_w\cdot\mathbf{v}_{c_i^-})\,\mathbf{v}_{c_i^-}$$

In [ ]:
def sigmoid(x):
    """Numerically stable sigmoid — clips to [-20, 20] to avoid overflow."""
    return 1 / (1 + np.exp(-np.clip(x, -20, 20)))


class Word2Vec:
    def __init__(self, vocab_size: int, embed_dim: int):
        self.V = vocab_size
        self.D = embed_dim
        # Xavier-style init: small random values
        self.W_in  = (np.random.rand(vocab_size, embed_dim) - 0.5) / embed_dim
        self.W_out = np.zeros((vocab_size, embed_dim))

    # ── Forward pass + loss ────────────────────────────────────────────────
    def forward(self, center_idx, context_idx, neg_indices):
        v_c   = self.W_in[center_idx]          # (D,)
        u_pos = self.W_out[context_idx]        # (D,)
        u_neg = self.W_out[neg_indices]        # (k, D)

        sig_pos = sigmoid(np.dot(v_c, u_pos))   # scalar in (0,1)
        sig_neg = sigmoid(u_neg @ v_c)           # (k,)

        eps  = 1e-7
        loss = -np.log(sig_pos + eps) - np.sum(np.log(1 - sig_neg + eps))

        # ── Gradients ─────────────────────────────────────────────────────
        # dL/dv_c
        grad_vc   = (sig_pos - 1) * u_pos + (sig_neg[:, None] * u_neg).sum(axis=0)
        # dL/du_pos
        grad_upos = (sig_pos - 1) * v_c
        # dL/du_neg_i  for each negative i
        grad_uneg = sig_neg[:, None] * v_c[None, :]  # (k, D)

        return loss, grad_vc, grad_upos, grad_uneg

    # ── SGD update ────────────────────────────────────────────────────────
    def update(self, center_idx, context_idx, neg_indices, lr):
        loss, grad_vc, grad_upos, grad_uneg = self.forward(
            center_idx, context_idx, neg_indices
        )
        self.W_in[center_idx]   -= lr * grad_vc
        self.W_out[context_idx] -= lr * grad_upos
        self.W_out[neg_indices] -= lr * grad_uneg
        return loss

    # ── Inference ─────────────────────────────────────────────────────────
    def get_embedding(self, word, vocab):
        return self.W_in[vocab.word2idx[word]]

    def cosine_similarity(self, w1, w2, vocab):
        v1 = self.get_embedding(w1, vocab)
        v2 = self.get_embedding(w2, vocab)
        return float(np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2) + 1e-8))

    def most_similar(self, word, vocab, topn=5):
        vec   = self.get_embedding(word, vocab)
        norms = np.linalg.norm(self.W_in, axis=1, keepdims=True) + 1e-8
        sims  = (self.W_in / norms) @ vec / (np.linalg.norm(vec) + 1e-8)
        sims[vocab.word2idx[word]] = -999
        top   = np.argsort(sims)[::-1][:topn]
        return [(vocab.idx2word[i], float(sims[i])) for i in top]

    def analogy(self, pos1, neg1, pos2, vocab, topn=3):
        """Vector arithmetic: pos1 - neg1 + pos2 ≈ answer (e.g. king-man+woman≈queen)"""
        if any(w not in vocab.word2idx for w in [pos1, neg1, pos2]):
            return []
        target = (self.get_embedding(pos1, vocab)
                  - self.get_embedding(neg1, vocab)
                  + self.get_embedding(pos2, vocab))
        norms  = np.linalg.norm(self.W_in, axis=1, keepdims=True) + 1e-8
        sims   = (self.W_in / norms) @ target / (np.linalg.norm(target) + 1e-8)
        for w in [pos1, neg1, pos2]:
            sims[vocab.word2idx[w]] = -999
        top = np.argsort(sims)[::-1][:topn]
        return [(vocab.idx2word[i], float(sims[i])) for i in top]


model = Word2Vec(vocab.size, embed_dim=50)
print(f"W_in  shape : {model.W_in.shape}   (center embeddings)")
print(f"W_out shape : {model.W_out.shape}  (context embeddings)")
print(f"Total parameters : {2 * vocab.size * 50:,}")

## 7 · Training Loop

**Learning rate schedule**: linear decay from `lr_start` → `lr_min` over all steps.  
This mirrors the original Word2Vec C code — high LR explores the loss landscape early; 
decay stabilises embeddings in the final epochs.

In [ ]:
EMBED_DIM   = 50
WINDOW      = 3
NEG_SAMPLES = 5
EPOCHS      = 300
LR_START    = 0.05
LR_MIN      = 0.0005

np.random.seed(42)
model       = Word2Vec(vocab.size, EMBED_DIM)
loss_history = []

for epoch in range(EPOCHS):
    # Fresh subsample + pairs each epoch
    sub   = subsample_tokens(tokens, vocab, t=1e-3)
    pairs = generate_skipgram_pairs(sub, vocab, window=WINDOW)
    np.random.shuffle(pairs)

    total_loss  = 0.0
    total_steps = len(pairs)

    for step, (center_idx, ctx_idx) in enumerate(pairs):
        progress   = (epoch * total_steps + step) / (EPOCHS * total_steps + 1)
        current_lr = max(LR_MIN, LR_START * (1 - progress))

        neg = neg_sampler.sample(NEG_SAMPLES, exclude=ctx_idx)
        total_loss += model.update(center_idx, ctx_idx, neg, current_lr)

    avg_loss = total_loss / (total_steps + 1e-8)
    loss_history.append(avg_loss)

    if (epoch + 1) % 30 == 0:
        print(f"Epoch {epoch+1:>3}/{EPOCHS}  |  loss={avg_loss:.4f}  |  lr={current_lr:.5f}")

print("\nTraining complete.")

## 8 · Training Loss Curve

In [ ]:
plt.figure(figsize=(9, 3))
plt.plot(loss_history, color='steelblue', linewidth=1.5)
plt.xlabel("Epoch")
plt.ylabel("Average Loss")
plt.title("Word2Vec Skip-gram + Negative Sampling — Training Loss")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 9 · Evaluation

### 9a · Nearest Neighbors

For each probe word, find the top-5 most similar words by cosine similarity.

In [ ]:
probe_words = ["king", "dog", "paris", "mountain", "machine", "fast"]

print("Nearest Neighbors (cosine similarity):")
print("─" * 60)
for word in probe_words:
    if word not in vocab.word2idx:
        print(f"  '{word}' not in vocab")
        continue
    neighbors = model.most_similar(word, vocab, topn=5)
    nbr_str   = "  |  ".join(f"{w} ({s:.2f})" for w, s in neighbors)
    print(f"  {word:<12} →  {nbr_str}")

### 9b · Pairwise Cosine Similarities

In [ ]:
pairs_to_check = [
    ("king",   "queen"),     # same semantic cluster → high
    ("dog",    "cat"),       # both pets → high
    ("paris",  "berlin"),    # both European capitals → high
    ("fast",   "slow"),      # antonyms, but share context → moderate-high
    ("king",   "mountain"),  # unrelated → low
]

print("Pairwise cosine similarities:")
print("─" * 45)
for w1, w2 in pairs_to_check:
    if w1 in vocab.word2idx and w2 in vocab.word2idx:
        sim = model.cosine_similarity(w1, w2, vocab)
        bar = "█" * int(abs(sim) * 20)
        print(f"  sim({w1:<10}, {w2:<10}) = {sim:+.4f}  {bar}")

### 9c · Vector Analogies

The famous **king − man + woman ≈ queen** result.

The geometry works because:
- `king − man` removes the "male royalty" vector from "male"
- Adding `woman` adds the "female" direction
- The result lands near "queen" in embedding space

In [ ]:
analogies = [
    ("king",   "man",    "woman"),    # expected: queen
    ("paris",  "france", "germany"),  # expected: berlin
    ("prince", "king",   "queen"),    # expected: princess
]

print("Vector Analogies  ( A − B + C ≈ ? )")
print("─" * 50)
for a, b, c in analogies:
    results = model.analogy(a, b, c, vocab, topn=3)
    if results:
        res_str = ", ".join(f"{w} ({s:.2f})" for w, s in results)
        print(f"  {a} − {b} + {c}  →  {res_str}")

## 10 · Embedding Visualisation (PCA)

Project all embeddings to 2D. Semantically related words should cluster together.

In [ ]:
# Colour-code word groups
word_groups = {
    "royalty"  : ["king", "queen", "prince", "princess", "man", "woman"],
    "animals"  : ["dog", "cat", "puppy", "kitten"],
    "capitals" : ["paris", "berlin", "london", "rome"],
    "countries": ["france", "germany", "england", "italy"],
    "nature"   : ["mountain", "river", "ocean", "forest", "valley"],
    "tech"     : ["machine", "learning", "neural", "networks", "algorithms"],
}
colors = ["#e63946", "#2a9d8f", "#457b9d", "#f4a261", "#6a4c93", "#2d6a4f"]

# Collect all words that are in vocab
all_words, all_vecs, word_color = [], [], []
for (group, words), color in zip(word_groups.items(), colors):
    for w in words:
        if w in vocab.word2idx:
            all_words.append(w)
            all_vecs.append(model.get_embedding(w, vocab))
            word_color.append(color)

vecs_arr = np.array(all_vecs)
coords   = PCA(n_components=2).fit_transform(vecs_arr)

fig, ax = plt.subplots(figsize=(12, 9))
# Draw group legend
for (group, _), color in zip(word_groups.items(), colors):
    ax.scatter([], [], color=color, label=group, s=80)
ax.legend(loc="upper left", fontsize=9)

ax.scatter(coords[:, 0], coords[:, 1], c=word_color, s=60, alpha=0.85, zorder=3)
for i, word in enumerate(all_words):
    ax.annotate(word, coords[i], fontsize=8.5,
                xytext=(5, 4), textcoords="offset points")

ax.set_title("Word2Vec Embeddings — PCA (2D projection)", fontsize=13)
ax.axis("off")
plt.tight_layout()
plt.show()

## Summary

### What we built
| Component | Implementation detail |
|-----------|----------------------|
| **Skip-gram** | For each center word, predict all context words within a dynamic window |
| **Negative Sampling** | Binary classification: positive pair vs. k=5 noise pairs |
| **Neg sampling dist** | Smoothed unigram freq^0.75 — biased but not dominated by common words |
| **Subsampling** | Frequent words discarded with P ∝ 1 − √(t/freq) each epoch |
| **Two matrices** | W_in (center) + W_out (context) — prevents trivial self-similarity |
| **Manual gradients** | Sigmoid BCE → ∂L/∂v analytically derived, no autograd |
| **LR schedule** | Linear decay lr_start → lr_min over total training steps |

### Why the embeddings work
The network is never told what "royalty" or "capital city" means.  
It only sees raw co-occurrence patterns.  
Yet words that appear in *similar contexts* end up in similar regions of ℝ^50 —  
and arithmetic over those regions recovers semantic relationships.

### Next steps to scale up
- Replace the hand-crafted corpus with Wikipedia / Common Crawl
- Increase `embed_dim` to 100–300
- Use mini-batches and vectorised updates for speed
- Add subword information (→ FastText)
- Use hierarchical softmax instead of negative sampling for very large vocabularies
